# imports

In [10]:
import os
import sys
import numpy as np
import pandas as pd
import mlflow
import mlflow.tensorflow

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# config

In [11]:
ASSET = 'BTC'
INTERVAL = '4h'
SEQUENCE_LENGTH = 20
N_FEATURES = 5
# model hyperparameters (same as old LSTM for fair comparison)
LSTM_UNITS_1 = 64
LSTM_UNITS_2 = 32
DENSE_UNITS = 16
DROPOUT_1 = 0.4
DROPOUT_2 = 0.2
LEARNING_RATE = 0.0001
EPOCHS = 50
BATCH_SIZE = 64
EARLY_STOP_PATIENCE = 15

DATA_DIR = '../../../data/processed/'

X_TRAIN_PATH = os.path.join(DATA_DIR, f'X_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
X_TEST_PATH  = os.path.join(DATA_DIR, f'X_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.npy')
Y_TRAIN_PATH = os.path.join(DATA_DIR, f'y_train_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')
Y_TEST_PATH  = os.path.join(DATA_DIR, f'y_test_{ASSET.lower()}_{INTERVAL}_lstm_raw.parquet')

print(f" Asset: {ASSET} | Interval: {INTERVAL}")
print(f" Sequence length: {SEQUENCE_LENGTH} candles | Features: {N_FEATURES} log-returns")
print(f" Input files:")
print(f"   X_train: {X_TRAIN_PATH}")
print(f"   X_test:  {X_TEST_PATH}")
print(f"   y_train: {Y_TRAIN_PATH}")
print(f"   y_test:  {Y_TEST_PATH}")


 Asset: BTC | Interval: 4h
 Sequence length: 20 candles | Features: 5 log-returns
 Input files:
   X_train: ../../../data/processed/X_train_btc_4h_lstm_raw.npy
   X_test:  ../../../data/processed/X_test_btc_4h_lstm_raw.npy
   y_train: ../../../data/processed/y_train_btc_4h_lstm_raw.parquet
   y_test:  ../../../data/processed/y_test_btc_4h_lstm_raw.parquet


# mlflow setup

In [12]:
mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = f"{ASSET}_{INTERVAL}_deep_learning"

try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
    print(f"Created new MLflow experiment: '{EXPERIMENT_NAME}'")
except:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
    print(f"Using existing MLflow experiment: '{EXPERIMENT_NAME}'")

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"   Experiment ID: {experiment_id}")

Using existing MLflow experiment: 'BTC_4h_deep_learning'
   Experiment ID: 21


# load pre-built sequences from fe notebook

In [13]:
X_train = np.load(X_TRAIN_PATH)
X_test  = np.load(X_TEST_PATH)

y_train_df = pd.read_parquet(Y_TRAIN_PATH)
y_test_df  = pd.read_parquet(Y_TEST_PATH)

y_train = y_train_df['target_direction'].values
y_test  = y_test_df['target_direction'].values

print(f"data loaded successfully")
print(f"   X_train shape: {X_train.shape}  (samples, timesteps, features)")
print(f"   X_test  shape: {X_test.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   y_test  shape: {y_test.shape}")


data loaded successfully
   X_train shape: (10619, 20, 5)  (samples, timesteps, features)
   X_test  shape: (2655, 20, 5)
   y_train shape: (10619,)
   y_test  shape: (2655,)


# verify data integrity

In [14]:
assert X_train.shape[1] == SEQUENCE_LENGTH, f"Expected {SEQUENCE_LENGTH} timesteps, got {X_train.shape[1]}"
assert X_train.shape[2] == N_FEATURES, f"Expected {N_FEATURES} features, got {X_train.shape[2]}"
assert X_test.shape[1] == SEQUENCE_LENGTH, f"Expected {SEQUENCE_LENGTH} timesteps, got {X_test.shape[1]}"
assert X_test.shape[2] == N_FEATURES, f"Expected {N_FEATURES} features, got {X_test.shape[2]}"
assert len(y_train) == X_train.shape[0], "y_train length mismatch"
assert len(y_test) == X_test.shape[0], "y_test length mismatch"

train_pos = y_train.mean()
test_pos  = y_test.mean()

print(f"All shape assertions passed")
print(f"   Train samples: {X_train.shape[0]:,}  |  Positive class: {train_pos:.1%}")
print(f"   Test  samples: {X_test.shape[0]:,}  |  Positive class: {test_pos:.1%}")
print(f"   Sequence: {SEQUENCE_LENGTH} candles × {N_FEATURES} log-return features")
print(f"   Feature columns: open_logret, high_logret, low_logret, close_logret, volume_logret")


All shape assertions passed
   Train samples: 10,619  |  Positive class: 51.5%
   Test  samples: 2,655  |  Positive class: 50.1%
   Sequence: 20 candles × 5 log-return features
   Feature columns: open_logret, high_logret, low_logret, close_logret, volume_logret


# model architecture (2-layer LSTM — same as old for fair comparison)

In [15]:
model = Sequential([
    LSTM(LSTM_UNITS_1, return_sequences=True, input_shape=(SEQUENCE_LENGTH, N_FEATURES)),
    Dropout(DROPOUT_1),

    LSTM(LSTM_UNITS_2, return_sequences=False),
    Dropout(DROPOUT_2),

    Dense(DENSE_UNITS, activation='relu'),
    Dropout(DROPOUT_2),

    Dense(1, activation='sigmoid')
])

print(f" Model architecture (same as old LSTM for direct comparison):")
print(f"   LSTM({LSTM_UNITS_1}, return_sequences=True) → Dropout({DROPOUT_1})")
print(f"   LSTM({LSTM_UNITS_2}, return_sequences=False) → Dropout({DROPOUT_2})")
print(f"   Dense({DENSE_UNITS}, relu) → Dropout({DROPOUT_2})")
print(f"   Dense(1, sigmoid)")
model.summary()

 Model architecture (same as old LSTM for direct comparison):
   LSTM(64, return_sequences=True) → Dropout(0.4)
   LSTM(32, return_sequences=False) → Dropout(0.2)
   Dense(16, relu) → Dropout(0.2)
   Dense(1, sigmoid)


c:\Users\Manindra\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                   │ (None, 20, 64)         │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,881 (120.63 KB)

 Trainable params: 30,881 (120.63 KB)

 Non-trainable params: 0 (0.00 B)

# compile model

In [16]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f" Model compiled:")
print(f"   Optimizer: Adam(lr={LEARNING_RATE})")
print(f"   Loss: binary_crossentropy")
print(f"   Metric: accuracy")

 Model compiled:
   Optimizer: Adam(lr=0.0001)
   Loss: binary_crossentropy
   Metric: accuracy


# early stopping callback

In [17]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOP_PATIENCE,
    restore_best_weights=True,
    verbose=1
)

print(f" EarlyStopping: patience={EARLY_STOP_PATIENCE}, monitor=val_loss, restore_best_weights=True")

 EarlyStopping: patience=15, monitor=val_loss, restore_best_weights=True


# train model with mlflow tracking

In [18]:
mlflow.tensorflow.autolog()

RUN_NAME = f"LSTM_raw_OHLCV_{ASSET}_{INTERVAL}_seq{SEQUENCE_LENGTH}"

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_param("asset", ASSET)
    mlflow.log_param("interval", INTERVAL)
    mlflow.log_param("sequence_length", SEQUENCE_LENGTH)
    mlflow.log_param("n_features", N_FEATURES)
    mlflow.log_param("feature_type", "log_returns_raw_ohlcv")
    mlflow.log_param("lstm_units_1", LSTM_UNITS_1)
    mlflow.log_param("lstm_units_2", LSTM_UNITS_2)
    mlflow.log_param("dense_units", DENSE_UNITS)
    mlflow.log_param("dropout_1", DROPOUT_1)
    mlflow.log_param("dropout_2", DROPOUT_2)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("early_stop_patience", EARLY_STOP_PATIENCE)
    mlflow.log_param("train_samples", X_train.shape[0])
    mlflow.log_param("test_samples", X_test.shape[0])
    mlflow.log_param("train_pos_pct", round(train_pos, 4))
    mlflow.log_param("test_pos_pct", round(test_pos, 4))

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stopping],
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

    mlflow.log_metric("test_loss", test_loss)
    mlflow.log_metric("test_accuracy", test_acc)

    print(f"\n Test Results: loss={test_loss:.4f}, accuracy={test_acc:.4f} ({test_acc*100:.2f}%)")

print(f"\n Training complete — run: '{RUN_NAME}'")

2026/05/22 13:27:02 WARNING mlflow.utils.autologging_utils: MLflow tensorflow autologging is known to be compatible with 2.16.2 <= tensorflow, but the installed version is 2.16.1. If you encounter errors during autologging, try upgrading / downgrading tensorflow to a compatible version, or try upgrading MLflow.


Epoch 1/50
130/133 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5039 - loss: 0.6931

133/133 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.5041 - loss: 0.6931 - val_accuracy: 0.5193 - val_loss: 0.6928
Epoch 2/50
132/133 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5176 - loss: 0.6928

133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5175 - loss: 0.6928 - val_accuracy: 0.5193 - val_loss: 0.6927
Epoch 3/50
130/133 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5171 - loss: 0.6926

133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5170 - loss: 0.6926 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 4/50
130/133 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5175 - loss: 0.6927

133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5174 - loss: 0.6927 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 5/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5118 - loss: 0.6929 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 6/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5148 - loss: 0.6929 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 7/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5090 - loss: 0.6932 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 8/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5140 - loss: 0.6928 - val_accuracy: 0.5193 - val_loss: 0.6925
Epoch 9/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5201 - loss: 0.6923 - val_accuracy: 0.5193 - val_loss: 0.6926
Epoch 10/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5092 - loss: 0.6930 - val_accuracy: 0.5193 - val_loss: 0.6926
Epoch 11/50
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.5086 - loss: 0.6930 - val_accuracy: 0.5

2026/05/22 13:27:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



 Test Results: loss=0.6934, accuracy=0.5009 (50.09%)
🏃 View run LSTM_raw_OHLCV_BTC_4h_seq20 at: http://localhost:5000/#/experiments/21/runs/627172b0038f43c2bb8d52d718c9180e
🧪 View experiment at: http://localhost:5000/#/experiments/21

 Training complete — run: 'LSTM_raw_OHLCV_BTC_4h_seq20'


# detailed evaluation on test set

In [19]:
y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print(f"{'='*60}")
print(f"   LSTM RAW OHLCV — Test Set Evaluation ({ASSET} {INTERVAL})")
print(f"{'='*60}")
print(f"  Accuracy:   {acc:.4f} ({acc*100:.2f}%)")
print(f"  Precision:  {prec:.4f} ({prec*100:.2f}%)")
print(f"  Recall:     {rec:.4f} ({rec*100:.2f}%)")
print(f"  F1 Score:   {f1:.4f} ({f1*100:.2f}%)")
print(f"{'='*60}")

OLD_LSTM_ACC = 0.5130
delta = (acc - OLD_LSTM_ACC) * 100
direction = " ABOVE" if delta > 0 else "▼ BELOW"
print(f"\n   Benchmark: Old LSTM (engineered features) = {OLD_LSTM_ACC*100:.2f}%")
print(f"   This model: {acc*100:.2f}% → {direction} benchmark by {abs(delta):.2f}pp")
print(f"{'='*60}")

   LSTM RAW OHLCV — Test Set Evaluation (BTC 4h)
  Accuracy:   0.5009 (50.09%)
  Precision:  0.5009 (50.09%)
  Recall:     1.0000 (100.00%)
  F1 Score:   0.6675 (66.75%)

   Benchmark: Old LSTM (engineered features) = 51.30%
   This model: 50.09% → ▼ BELOW benchmark by 1.21pp


# classification report & confusion matrix

In [20]:
print(" Classification Report:")
print(classification_report(y_test, y_pred, target_names=['DOWN (0)', 'UP (1)'], zero_division=0))

print(" Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(f"              Predicted DOWN  Predicted UP")
print(f"  Actual DOWN      {cm[0][0]:>6}        {cm[0][1]:>6}")
print(f"  Actual UP        {cm[1][0]:>6}        {cm[1][1]:>6}")

tn, fp, fn, tp = cm.ravel()
print(f"\n  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"  TPR (Recall): {tp/(tp+fn):.4f}" if (tp+fn) > 0 else "  TPR: N/A")
print(f"  TNR:          {tn/(tn+fp):.4f}" if (tn+fp) > 0 else "  TNR: N/A")

 Classification Report:
              precision    recall  f1-score   support

    DOWN (0)       0.00      0.00      0.00      1325
      UP (1)       0.50      1.00      0.67      1330

    accuracy                           0.50      2655
   macro avg       0.25      0.50      0.33      2655
weighted avg       0.25      0.50      0.33      2655

 Confusion Matrix:
              Predicted DOWN  Predicted UP
  Actual DOWN           0          1325
  Actual UP             0          1330

  TN=0  FP=1325  FN=0  TP=1330
  TPR (Recall): 1.0000
  TNR:          0.0000


# experiment conclusion: raw ohlcv vs engineered features

In [22]:
print(f"{'='*60}")
print(f"    CONCLUSION")
print(f"{'='*60}")
print(f"  Model: 2-layer LSTM on raw OHLCV log-returns (no indicators)")
print(f"  Data:  {ASSET} {INTERVAL}, {SEQUENCE_LENGTH}-candle sequences")
print(f"  Features: open, high, low, close, volume log-returns (5 total)")
print(f"")
print(f"  Old LSTM (engineered features): {OLD_LSTM_ACC*100:.2f}%")
print(f"  This LSTM (raw OHLCV only):     {acc*100:.2f}%")
print(f"  Delta: {delta:+.2f}pp")
print(f"")
if acc > OLD_LSTM_ACC:
    print(f"   RAW OHLCV BEATS engineered features!")
    print(f"  → Next step: LSTM+XGBoost hybrid (Experiment #98)")
else:
    print(f"   Raw OHLCV does NOT beat engineered features")
    print(f"  → Raw prices alone are insufficient for LSTM")
    print(f"  → Hybrid LSTM+XGBoost likely won't help either")
print(f"{'='*60}")


    CONCLUSION
  Model: 2-layer LSTM on raw OHLCV log-returns (no indicators)
  Data:  BTC 4h, 20-candle sequences
  Features: open, high, low, close, volume log-returns (5 total)

  Old LSTM (engineered features): 51.30%
  This LSTM (raw OHLCV only):     50.09%
  Delta: -1.21pp

   Raw OHLCV does NOT beat engineered features
  → Raw prices alone are insufficient for LSTM
  → Hybrid LSTM+XGBoost likely won't help either
